# Primary dataset: player weekly stats

Source: [nflverse](https://github.com/nflverse/nflreadpy), accessed via the `nflreadpy` package.

One row = one player's stat line for one week of one season. Includes `rushing_*`
columns (rushing_yards, rushing_tds, rushing_epa, carries, ...) and `passing_*`
columns (passing_yards, passing_tds, passing_epa, attempts, ...) side by side,
so this is the main dataset for comparing rushing vs. passing production
at the player level.

In [1]:
import polars as pl
import nflreadpy as nfl

In [2]:
# Adjust seasons as the group settles on scope.
player_stats = nfl.load_player_stats([2021, 2022, 2023, 2024, 2025])
player_stats.shape

(94848, 150)

In [3]:
player_stats.head()

player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,passing_10,passing_16,passing_20,passing_40,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,…,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,pt_att,pt_blocked,pt_long,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
str,str,str,str,str,str,i32,i32,str,str,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,str,str,str,i32,i32,i32,i32,i32,i32,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64
"""00-0019596""","""T.Brady""","""Tom Brady""","""QB""","""QB""","""https://static.www.nfl.com/ima…",2021,1,"""REG""","""2021_01_DAL_TB""","""TB""","""DAL""",32,50,379,4,2,0,0,0,0,446,176,22,14.006987,3.231739,0,0.849776,15,10,6,1,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,27.16,27.16
"""00-0022824""","""A.Lee""","""Andy Lee""","""P""","""SPEC""","""https://static.www.nfl.com/ima…",2021,1,"""REG""","""2021_01_ARI_TEN""","""ARI""","""TEN""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,3,0,58,144,2,0,1,1,1,0,0,0,124,0.0,0.0
"""00-0022924""","""B.Roethlisberger""","""Ben Roethlisberger""","""QB""","""QB""","""https://static.www.nfl.com/ima…",2021,1,"""REG""","""2021_01_PIT_BUF""","""PIT""","""BUF""",18,32,188,1,0,2,-11,1,0,196,85,9,-0.827915,-10.494156,0,0.959184,8,4,3,0,4,5,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,12.02,12.02
"""00-0023252""","""R.Gould""","""Robbie Gould""","""K""","""SPEC""","""https://static.www.nfl.com/ima…",2021,1,"""REG""","""2021_01_SF_DET""","""SF""","""DET""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,1,0,"""40;52""","""52""",null,92,52,0,5,5,0,0,1.0,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0023459""","""Aa.Rodgers""","""Aaron Rodgers""","""QB""","""QB""","""https://static.www.nfl.com/ima…",2021,1,"""REG""","""2021_01_GB_NO""","""GB""","""NO""",15,28,133,0,2,1,-11,0,0,238,72,6,-15.949715,-8.025815,0,0.558824,4,2,1,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,1.32,1.32


In [4]:
# Columns relevant to rushing vs. passing
cols = [c for c in player_stats.columns if c.startswith('rushing_') or c.startswith('passing_')]
cols

['passing_yards',
 'passing_tds',
 'passing_interceptions',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'passing_10',
 'passing_16',
 'passing_20',
 'passing_40',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'rushing_10',
 'rushing_12',
 'rushing_20',
 'rushing_40']

In [5]:
player_stats.select(cols).describe()

statistic,passing_yards,passing_tds,passing_interceptions,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,passing_10,passing_16,passing_20,passing_40,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,rushing_10,rushing_12,rushing_20,rushing_40
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",94848.0,94848.0,94848.0,94848.0,94848.0,94848.0,3528.0,3449.0,94848.0,94848.0,94848.0,94848.0,94848.0,94848.0,94848.0,94848.0,94848.0,94848.0,11871.0,94848.0,94848.0,94848.0,94848.0,94848.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,91320.0,91399.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,82977.0,0.0,0.0,0.0,0.0,0.0
"""mean""",7.066865,0.044071,0.022699,7.744781,3.365796,0.338668,0.680623,-0.309063,0.002362,0.279647,0.139096,0.088921,0.014254,3.517375,0.027486,0.011007,0.005008,0.200879,-0.214028,0.00116,0.091177,0.064324,0.019241,0.003574
"""std""",41.377076,0.314675,0.19778,45.854229,20.192499,1.999977,9.551509,16.928497,0.050457,1.662565,0.863762,0.581212,0.142675,14.746988,0.19208,0.11128,0.071333,0.837748,2.722811,0.034344,0.448956,0.346668,0.157134,0.062271
"""min""",-7.0,0.0,0.0,-17.0,0.0,0.0,-35.56482,-89.496177,0.0,0.0,0.0,0.0,0.0,-34.0,0.0,0.0,0.0,0.0,-15.917273,0.0,0.0,0.0,0.0,0.0
"""25%""",0.0,0.0,0.0,0.0,0.0,0.0,-4.867112,-6.131655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.475773,0.0,0.0,0.0,0.0,0.0
"""50%""",0.0,0.0,0.0,0.0,0.0,0.0,0.309899,0.483702,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.218875,0.0,0.0,0.0,0.0,0.0
"""75%""",0.0,0.0,0.0,0.0,0.0,0.0,6.661802,7.473331,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.05567,0.0,0.0,0.0,0.0,0.0
"""max""",525.0,6.0,5.0,633.0,280.0,27.0,33.188525,72.446629,2.0,26.0,14.0,11.0,4.0,255.0,4.0,4.0,2.0,15.0,17.356537,2.0,8.0,7.0,4.0,3.0


In [6]:
player_stats.dtypes

[String,
 String,
 String,
 String,
 String,
 String,
 Int32,
 Int32,
 String,
 String,
 String,
 String,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Float64,
 Float64,
 Int32,
 Float64,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Float64,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Float64,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Float64,
 Float64,
 Float64,
 Float64,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Float64,
 Float64,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Float64,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 Int32,
 String,
 Stri